In [6]:
from pathlib import Path

PROJECT_ROOT = Path(r"C:\CN\AI-Adaptive-Routing")

MODEL_PATH = PROJECT_ROOT / "ml" / "route_cost_model.pkl"
DATA_PATH = PROJECT_ROOT / "dataset" / "processed" / "flow_analysis.csv"

print("Project folder exists:", PROJECT_ROOT.exists())
print("Model exists:", MODEL_PATH.exists())
print("Dataset exists:", DATA_PATH.exists())

Project folder exists: True
Model exists: True
Dataset exists: True


In [7]:

from pathlib import Path
import joblib
import pandas as pd
import numpy as np
from IPython.display import display

# Project paths
ROOT = Path(r"C:\CN\AI-Adaptive-Routing")
MODEL_PATH = ROOT / "ml" / "route_cost_model.pkl"
DATA_PATH = ROOT / "dataset" / "processed" / "flow_analysis.csv"
NS3_PATH = ROOT / "results" / "csv" / "network_data.csv"

# 1. Load trained model
package = joblib.load(MODEL_PATH)
model = package["model"]
features = package["features"]
target = package["target"]

print("MODEL:", type(model).__name__)
print("FEATURES:", features)
print("TARGET:", target)

# 2. Load dataset and predict costs
df = pd.read_csv(DATA_PATH)

missing = [col for col in features if col not in df.columns]
if missing:
    raise ValueError(f"Dataset is missing model features: {missing}")

X = df[features].copy()
df["predicted_route_cost"] = model.predict(X)
df["prediction_error"] = df["predicted_route_cost"] - df[target]

mae = df["prediction_error"].abs().mean()

print(f"\nDataset rows: {len(df)}")
print(f"Prediction MAE: {mae:.6f}")
display(df[
    ["flow_id", "source", "destination",
     target, "predicted_route_cost", "prediction_error"]
].head(10))

# 3. Inspect actual NS-3 output without modifying it
print("\n--- NS-3 DATA CHECK ---")

if NS3_PATH.exists():
    ns3_df = pd.read_csv(NS3_PATH)
    print("NS-3 file:", NS3_PATH)
    print("Rows:", len(ns3_df))
    print("Columns:", ns3_df.columns.tolist())
    display(ns3_df.head(5))

    missing_ns3 = [col for col in features if col not in ns3_df.columns]
    print("\nFeatures missing from NS-3 CSV:", missing_ns3)
else:
    print("network_data.csv not found at:", NS3_PATH)
    print("We'll inspect the available NS-3 output files next.")

print("\nCheck complete. No existing file was overwritten.")


MODEL: DecisionTreeRegressor
FEATURES: ['packets_sent', 'packets_received', 'packets_lost', 'delay', 'throughput', 'jitter', 'packet_delivery_ratio_percent']
TARGET: route_cost

Dataset rows: 9
Prediction MAE: 0.004160


,flow_id,source,destination,route_cost,predicted_route_cost,prediction_error
0,1,10.1.2.2,10.1.9.2,0.299475,0.299475,0.000000
1,2,10.1.1.2,10.1.10.2,0.336913,0.299475,-0.037438
2,3,10.1.9.2,10.1.2.2,0.372506,0.372506,0.000000
3,1,10.1.2.2,10.1.9.2,0.077082,0.077082,0.000000
4,2,10.1.1.2,10.1.10.2,0.118361,0.118361,0.000000
5,3,10.1.9.2,10.1.2.2,0.361980,0.361980,0.000000
6,1,10.1.2.2,10.1.9.2,0.077079,0.077079,0.000000
7,2,10.1.1.2,10.1.10.2,0.118361,0.118361,0.000000
8,3,10.1.9.2,10.1.2.2,0.356948,0.356948,0.000000



--- NS-3 DATA CHECK ---
network_data.csv not found at: C:\CN\AI-Adaptive-Routing\results\csv\network_data.csv
We'll inspect the available NS-3 output files next.

Check complete. No existing file was overwritten.


## NS-3 output locate

In [8]:

from pathlib import Path
import pandas as pd

ROOT = Path(r"C:\CN\AI-Adaptive-Routing")
NS3_ROOT = Path(r"C:\ns3\ns-3.47")

print("=== PROJECT CSV FILES ===")
for folder in [
    ROOT / "results" / "csv",
    ROOT / "dataset" / "processed",
    NS3_ROOT
]:
    print(f"\nFolder: {folder}")
    if folder.exists():
        files = list(folder.glob("*.csv"))
        if files:
            for file in files:
                print(f"  {file.name} | {file.stat().st_size} bytes")
        else:
            print("  No CSV files found")
    else:
        print("  Folder not found")

print("\n=== NS-3 FLOW DATA PREVIEW ===")
for file in [
    ROOT / "results" / "csv" / "network_data.csv",
    NS3_ROOT / "network_data.csv"
]:
    if file.exists():
        df_ns3 = pd.read_csv(file)
        print("\nFound:", file)
        print("Rows:", len(df_ns3))
        print("Columns:", df_ns3.columns.tolist())
        display(df_ns3.head())


=== PROJECT CSV FILES ===

Folder: C:\CN\AI-Adaptive-Routing\results\csv
  ai_routing_engine_predictions.csv | 992 bytes
  route_cost_predictions.csv | 1062 bytes

Folder: C:\CN\AI-Adaptive-Routing\dataset\processed
  all_flows.csv | 710 bytes
  all_monitoring.csv | 1038 bytes
  flow_analysis.csv | 885 bytes

Folder: C:\ns3\ns-3.47
  adaptive_route_log.csv | 129 bytes
  monitoring_data.csv | 617 bytes
  network_data.csv | 286 bytes

=== NS-3 FLOW DATA PREVIEW ===

Found: C:\ns3\ns-3.47\network_data.csv
Rows: 3
Columns: ['flow_id', 'source', 'destination', 'packets_sent', 'packets_received', 'packets_lost', 'delay', 'throughput', 'jitter']


,flow_id,source,destination,packets_sent,packets_received,packets_lost,delay,throughput,jitter
0,1,10.1.2.2,10.1.9.2,17919,17913,6,0.054214,4.925140,4.961110e-04
1,2,10.1.1.2,10.1.10.2,43945,22494,12789,3.909870,9.963930,1.262130e+00
2,3,10.1.9.2,10.1.2.2,9353,9353,0,0.019140,0.231694,2.737380e-08


## Actual NS-3 data par ML

In [9]:

from pathlib import Path
import joblib
import pandas as pd
import numpy as np
from IPython.display import display

ROOT = Path(r"C:\CN\AI-Adaptive-Routing")
NS3_CSV = Path(r"C:\ns3\ns-3.47\network_data.csv")
MODEL_PATH = ROOT / "ml" / "route_cost_model.pkl"
OUTPUT = ROOT / "results" / "csv" / "ns3_ml_predictions.csv"

# Load real NS-3 flow data and trained model
df = pd.read_csv(NS3_CSV)
package = joblib.load(MODEL_PATH)
model = package["model"]
features = package["features"]

# Calculate derived metrics from measured NS-3 data
df["packet_delivery_ratio_percent"] = (
    df["packets_received"] / df["packets_sent"].replace(0, np.nan) * 100
).fillna(0)

loss_rate = (
    df["packets_lost"] / df["packets_sent"].replace(0, np.nan)
).fillna(0)

# Same route-cost formula used in the existing project
df["route_cost"] = (
    0.40 * (df["delay"] / 0.1).clip(0, 1)
    + 0.30 * loss_rate.clip(0, 1)
    + 0.30 * (1 - (df["throughput"] / 10).clip(0, 1))
)

# Predict using the trained ML model
missing = [col for col in features if col not in df.columns]
if missing:
    raise ValueError(f"Missing required features: {missing}")

df["predicted_route_cost"] = model.predict(df[features])

# Save separately; original files remain unchanged
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT, index=False)

print("Real NS-3 flows:", len(df))
print("Model features:", features)
print("Saved predictions to:", OUTPUT)
display(df[[
    "flow_id", "source", "destination",
    "delay", "throughput", "packets_lost",
    "packet_delivery_ratio_percent",
    "route_cost", "predicted_route_cost"
]].round(6))


Real NS-3 flows: 3
Model features: ['packets_sent', 'packets_received', 'packets_lost', 'delay', 'throughput', 'jitter', 'packet_delivery_ratio_percent']
Saved predictions to: C:\CN\AI-Adaptive-Routing\results\csv\ns3_ml_predictions.csv


,flow_id,source,destination,delay,throughput,packets_lost,packet_delivery_ratio_percent,route_cost,predicted_route_cost
0,1,10.1.2.2,10.1.9.2,0.054214,4.925140,6,99.966516,0.369203,0.299475
1,2,10.1.1.2,10.1.10.2,3.909870,9.963930,12789,51.186711,0.488389,0.077082
2,3,10.1.9.2,10.1.2.2,0.019140,0.231694,0,100.000000,0.369609,0.372506


In [10]:

from pathlib import Path
import shutil

NS3_ROOT = Path(r"C:\ns3\ns-3.47")
OUT = Path(r"C:\CN\AI-Adaptive-Routing\results\csv")
OUT.mkdir(parents=True, exist_ok=True)

files = {
    "network_data.csv": "adaptive_high_network_data.csv",
    "monitoring_data.csv": "adaptive_high_monitoring_data.csv",
    "adaptive_route_log.csv": "adaptive_high_route_log.csv",
}

for source_name, dest_name in files.items():
    source = NS3_ROOT / source_name
    dest = OUT / dest_name

    if source.exists():
        shutil.copy2(source, dest)
        print("Saved:", dest)
    else:
        print("Not found:", source)

print("Current simulation results archived.")



Saved: C:\CN\AI-Adaptive-Routing\results\csv\adaptive_high_network_data.csv
Saved: C:\CN\AI-Adaptive-Routing\results\csv\adaptive_high_monitoring_data.csv
Saved: C:\CN\AI-Adaptive-Routing\results\csv\adaptive_high_route_log.csv
Current simulation results archived.


In [11]:

from pathlib import Path
import pandas as pd

NS3 = Path(r"C:\ns3\ns-3.47")

for name in ["network_data.csv", "monitoring_data.csv", "adaptive_route_log.csv"]:
    path = NS3 / name
    print(f"\n--- {name} ---")
    if path.exists():
        df = pd.read_csv(path)
        print("Rows:", len(df))
        print("Columns:", df.columns.tolist())
        display(df.head(3))
    else:
        print("File not found:", path)




--- network_data.csv ---
Rows: 3
Columns: ['flow_id', 'source', 'destination', 'packets_sent', 'packets_received', 'packets_lost', 'delay', 'throughput', 'jitter']


,flow_id,source,destination,packets_sent,packets_received,packets_lost,delay,throughput,jitter
0,1,10.1.2.2,10.1.9.2,17919,17913,6,0.054214,4.925140,4.961110e-04
1,2,10.1.1.2,10.1.10.2,43945,22494,12789,3.909870,9.963930,1.262130e+00
2,3,10.1.9.2,10.1.2.2,9353,9353,0,0.019140,0.231694,2.737380e-08



--- monitoring_data.csv ---
Rows: 20
Columns: ['time', 'total_queue_packets', 'average_queue_packets', 'utilization_percent']


,time,total_queue_packets,average_queue_packets,utilization_percent
0,2.0,0,0.0000,10.0
1,3.0,0,0.0000,10.0
2,4.0,100,2.7778,10.0



--- adaptive_route_log.csv ---
Rows: 5
Columns: ['time', 'queue_A', 'queue_B', 'selected_path']


,time,queue_A,queue_B,selected_path
0,10,100,0,Path_B
1,12,200,200,Path_A
2,14,200,200,Path_A
